<a href="https://colab.research.google.com/github/IT24101853-AkashD/-SE3082_Lab09_IT24101853/blob/main/SE3082_Lab09_IT24101853.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Mon Oct 12 09:33:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [3]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

name, compute_cap
Tesla T4, 7.5


In [7]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void) {
    hello<<<1, 4>>>();       // 1 block of 4 threads
    cudaDeviceSynchronize(); // wait so the GPU's output is flushed
    return 0;
}

Writing hello.cu


1. Exercise 1 Checklist & Questions (Page 3)
GPU Name: NVIDIA Tesla T4
Compute Capability: 7.5 (-arch=sm_75)
CUDA Version: CUDA 12.x

In [8]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello


Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


## Exercise 2 – Adding two numbers on the GPU

In [9]:
%%writefile ex2_add.cu
#include <cstdio>

// Kernel: this code runs on the device (GPU)
__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

// host = CPU, device = GPU
int main(void) {
    int a, b, c;            // host copies of a, b, c
    int *d_a, *d_b, *d_c;   // device copies of a, b, c
    int size = sizeof(int);

    // Allocate space on the GPU for d_a, d_b, d_c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Set input values on CPU
    a = 2;
    b = 7;

    // 1. Copy inputs from CPU (host) to GPU (device)
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // 2. Launch kernel on GPU: 1 block, 1 thread
    add<<<1, 1>>>(d_a, d_b, d_c);

    // Check for kernel launch errors
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // 3. Copy result back from GPU (device) to CPU (host)
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result is %d\n", c);

    // Free GPU memory
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex2_add.cu


In [10]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add

Result is 9


## Exercise 3 – Vector addition with 512 blocks and one thread each

In [11]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;         // host copies of a, b, c
    int *d_a, *d_b, *d_c;   // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space on host and fill with random numbers
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel with N (512) blocks, 1 thread per block
    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex3_blocks.cu


In [12]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
35 + 57 = 92
86 + 14 = 100
92 + 68 = 160
49 + 5 = 54
21 + 58 = 79
62 + 12 = 74
27 + 86 = 113
90 + 0 = 90
59 + 46 = 105
63 + 26 = 89
26 + 94 = 120
40 + 16 = 56
26 + 52 = 78
72 + 78 = 150
36 + 29 = 65
11 + 46 = 57
68 + 90 = 158
67 + 47 = 114
29 + 70 = 99
82 + 51 = 133
30 + 80 = 110
62 + 31 = 93
23 + 93 = 116
67 + 57 = 124
35 + 27 = 62
29 + 12 = 41
2 + 86 = 88
22 + 14 = 36
58 + 55 = 113
69 + 12 = 81
67 + 90 = 157
93 + 12 = 105
56 + 79 = 135
11 + 10 = 21
42 + 69 = 111
29 + 89 = 118
73 + 74 = 147
21 + 55 = 76
19 + 41 = 60
84 + 20 = 104
37 + 33 = 70
98 + 87 = 185
24 + 88 = 112
15 + 38 = 53
70 + 66 = 136
13 + 70 = 83
26 + 84 = 110
91 + 56 = 147
80 + 17 = 97
56 + 6 = 62
73 + 60 = 133
62 + 49 = 111
70 + 37 = 107
96 + 5 = 101
81 + 59 = 140
5 + 17 = 22
25 + 18 = 43
84 + 45 = 129
27 + 83 = 110
36 + 73 = 109
5 + 58 = 63
46 + 73 = 119
29 + 37 = 66
13 + 89 = 102
57 + 83 = 140
24 + 7 = 31
95 + 78 = 173
82 + 57 = 139
45 + 14 = 59
14 +

## Exercise 4 – Vector addition using one block and 512 threads

In [13]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    // each thread computes one element using threadIdx.x
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;         // host copies of a, b, c
    int *d_a, *d_b, *d_c;   // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch addT() kernel with 1 block of N (512) threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    // Check GPU answers against CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex4_threads.cu


In [14]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
5) 35 + 57 = 92
6) 86 + 14 = 100
7) 92 + 68 = 160
8) 49 + 5 = 54
9) 21 + 58 = 79
10) 62 + 12 = 74
11) 27 + 86 = 113
12) 90 + 0 = 90
13) 59 + 46 = 105
14) 63 + 26 = 89
15) 26 + 94 = 120
16) 40 + 16 = 56
17) 26 + 52 = 78
18) 72 + 78 = 150
19) 36 + 29 = 65
20) 11 + 46 = 57
21) 68 + 90 = 158
22) 67 + 47 = 114
23) 29 + 70 = 99
24) 82 + 51 = 133
25) 30 + 80 = 110
26) 62 + 31 = 93
27) 23 + 93 = 116
28) 67 + 57 = 124
29) 35 + 27 = 62
30) 29 + 12 = 41
31) 2 + 86 = 88
32) 22 + 14 = 36
33) 58 + 55 = 113
34) 69 + 12 = 81
35) 67 + 90 = 157
36) 93 + 12 = 105
37) 56 + 79 = 135
38) 11 + 10 = 21
39) 42 + 69 = 111
40) 29 + 89 = 118
41) 73 + 74 = 147
42) 21 + 55 = 76
43) 19 + 41 = 60
44) 84 + 20 = 104
45) 37 + 33 = 70
46) 98 + 87 = 185
47) 24 + 88 = 112
48) 15 + 38 = 53
49) 70 + 66 = 136
50) 13 + 70 = 83
51) 26 + 84 = 110
52) 91 + 56 = 147
53) 80 + 17 = 97
54) 56 + 6 = 62
55) 73 + 60 = 133
56) 62 + 49 = 111
57) 70 + 37 = 

## Exercise 5 – Multiplying two 10,000,000-element vectors

In [15]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000 // 10 million elements
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: global index of this thread
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: only compute if i is inside the array
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;         // host copies of a, b, c
    int *d_a, *d_b, *d_c;   // device copies of a, b, c
    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate inputs
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: number of blocks needed to cover N elements (round UP)
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    // TODO 4: launch vecMul with <<<blocks, THREADS_PER_BLOCK>>>
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 5: copy the result d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 6: print the last 1000 results as "index) a x b = c"
    for (int r = N - 1000; r < N; r++) {
        printf("%d) %d x %d = %d\n", r, a[r], b[r], c[r]);
    }

    // TODO 7: verify all N results on the CPU and print mismatches
    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] * b[r]) {
            errors++;
        }
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex5_vecmul.cu


In [16]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999001) 76 x 74 = 5624
9999002) 41 x 94 = 3854
9999003) 8 x 0 = 0
9999004) 1 x 41 = 41
9999005) 28 x 52 = 1456
9999006) 45 x 4 = 180
9999007) 63 x 35 = 2205
9999008) 89 x 97 = 8633
9999009) 10 x 71 = 710
9999010) 74 x 91 = 6734
9999011) 65 x 41 = 2665
9999012) 29 x 60 = 1740
9999013) 53 x 5 = 265
9999014) 35 x 88 = 3080
9999015) 18 x 12 = 216
9999016) 17 x 12 = 204
9999017) 8 x 65 = 520
9999018) 51 x 34 = 1734
9999019) 79 x 64 = 5056
9999020) 13 x 89 = 1157
9999021) 10 x 85 = 850
9999022) 7 x 2 = 14
9999023) 31 x 51 = 1581
9999024) 88 x 86 = 7568
9999025) 59 x 8 = 472
9999026) 94 x 50 = 4700
9999027) 58 x 94 = 5452
9999028) 5 x 53 = 265
9999029) 36 x 67 = 2412
9999030) 80 x 45 = 3600
9999031) 33 x 5 = 165
9999032) 64 x 41 = 2624
9999033) 21 x 91 = 1911
9999034) 93 x 6 = 558
9999035) 65 x 34 = 2210
9999036) 1 x 43 = 43
9999037) 90 x 62 = 5580
9999038) 80 x 21 = 1680
9999039) 90 x 92 = 8280
9999040) 1 x 86 = 86
9999041) 54 x 1

# Lab Sheet 9 – Conceptual Questions & Answers Summary

---

## Exercise 1 – Environment Checklist
- **GPU Model:** NVIDIA Tesla T4
- **Compute Capability:** 7.5 (compiler flag: `-arch=sm_75`)
- **CUDA Release:** CUDA 12.x

---

## Exercise 2 – THINK Questions
1. **Why do we need separate variables `d_a, d_b, d_c` on the device instead of using `a, b, c` directly?**
   - **Answer:** The CPU (host) and GPU (device) have physically separate memory spaces (System RAM vs GPU VRAM). The GPU hardware cannot directly dereference host CPU memory addresses. Therefore, device memory must be explicitly allocated on the GPU using `cudaMalloc` and data copied over via the PCIe bus using `cudaMemcpy`.

2. **In `cudaMemcpy(d_a, &a, ...)` we pass `&a`, but in the kernel launch we pass `d_a`. Why?**
   - **Answer:** `a` is a local integer on the CPU stack, so `&a` provides its CPU memory address to copy data from. On the other hand, `d_a` is already a pointer variable holding the GPU memory address allocated by `cudaMalloc`, which the GPU kernel requires directly.

3. **How many threads run the add kernel? What would you change to run it with 10 threads, and would the answer change?**
   - **Answer:** Exactly 1 thread runs the kernel (`<<<1, 1>>>`). To run 10 threads, the launch configuration would be changed to `add<<<1, 10>>>(d_a, d_b, d_c)`. All 10 threads execute `*c = *a + *b` in parallel and write 9 to `*c` simultaneously. The result is still 9, but the redundant parallel writes are inefficient.

---

## Exercise 3 – PREDICT BEFORE YOU RUN
- **Question:** *How many blocks and how many threads in total does `add<<<N, 1>>>` launch? A GPU executes threads in groups of 32 called warps. What does that mean for a block that contains only one thread?*
- **Answer:** It launches **512 blocks** and **512 total threads** (1 thread per block). Because NVIDIA GPUs schedule and execute instructions in hardware groups of 32 threads (warps), each block is allocated a full 32-thread warp where **only 1 thread is active and 31 threads remain completely idle**. This wastes approximately 97% of GPU warp compute capacity.

---

## Exercise 4 – EXPERIMENT (Find the Hardware Limit)
- **Question:** *What happens when `#define N 2048` is set? Which part of the program told you, and which GPU limit did you hit?*
- **Answer:**
  - **Output message:** `Kernel launch failed: invalid configuration argument`
  - **Program component:** `cudaGetLastError()` detected the failure and `cudaGetErrorString()` printed the readable error message.
  - **GPU Limit Hit:** The hardware architectural limit of **1024 maximum threads per block** on NVIDIA GPUs. Because 2048 exceeds 1024, the GPU rejects the kernel launch configuration.

---

## Exercise 5 – THINK & Bounds Checking
1. **Why is the boundary guard `if (i < n)` necessary?**
   - **Answer:** The launch configuration launches 19,532 blocks × 512 threads = **10,000,384 threads**, which is 384 threads more than the 10,000,000 array elements. Without the `if (i < n)` guard, the trailing 384 threads would attempt to read and write beyond the bounds of the allocated memory, causing undefined behavior or crashes.

2. **Why is it a good idea to use a multiple of 32 for threads per block? What happens if you use 500 instead of 512?**
   - **Answer:** GPU instructions are executed in warps of 32 threads. If 500 threads per block are used, the GPU still must schedule $\lceil 500 / 32 \rceil = 16$ warps (512 slots), leaving 12 thread slots per block permanently idle and wasted. Using multiples of 32 ensures full warp occupancy and 100% hardware efficiency.

---

## Exercise 6 – 2D Grid Mapping Summary
- **Grid Configuration:** `dim3 numBlocks(313, 625)` with `dim3 threadsPerBlock(32, 16)` = **195,625 blocks** (100,160,000 threads).
- **Index Calculations:**
  - Column: `col = blockIdx.x * blockDim.x + threadIdx.x;`
  - Row: `row = blockIdx.y * blockDim.y + threadIdx.y;`
  - Flat 1D Index: `idx = row * cols + col;`
- **Bounds Check:** `if (row < rows && col < cols)` safely guards against out-of-bounds indices in both dimensions.